### 덕덕고 API를 통해 ai가 검색 후 답변하는 기능 추가
- GPT 모델 자체는 학습한 시점까지의 정보만 갖고있기 때문에 최신정보에 정확한 답변을 기대하기 어렵다.
- 참고로 학습일 기준 OpenAI의 가장 최신 플래그쉽 모델은 **GPT-6 Astra**이다.

In [10]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

model = ChatOpenAI(model="gpt-5.6-luna")
display(Markdown(model.invoke("OpenAI의 가장 최신 모델은 무엇인가요?").content))

현재 **2026년 9월 기준 OpenAI의 최신 모델**은 실시간 모델 목록을 확인해야 정확히 답할 수 있습니다. 저는 현재 OpenAI의 최신 출시 정보를 실시간으로 조회할 수 없으므로 특정 모델명을 단정하기 어렵습니다.

가장 정확한 확인 방법은 다음과 같습니다.

- **ChatGPT:** 모델 선택 메뉴에서 최상위 모델 확인
- **API:** [OpenAI 플랫폼 모델 목록](https://platform.openai.com/docs/models)
- **공식 발표:** [OpenAI 뉴스](https://openai.com/news)

참고로 2025년 이후에는 **GPT-5 계열**이 최신 세대 모델로 알려져 있지만, 현재는 그 후속 버전이나 변형 모델이 출시되었을 수 있습니다.

- 덕덕고 검색 패키지 설치 및 테스트

In [ ]:
%uv add ddgs

In [18]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke("OpenAI의 가장 최신 모델은 무엇인가요?")

print(docs)

snippet: May 28, 2026 - 이들 각각은 증가된 크기(훈련 가능한 매개변수의 수) 및 훈련으로 인해 이전보다 훨씬 더 많은 능력을 발휘했다. 가장 최근인 GPT-4는 2023년 3월에 출시되었다. 이러한 모델은 지침을 따르도록 미세 조정된 모델을 포함하여 ..., title: GPT (언어 모델) - 위키백과, 우리 모두의 백과사전, link: https://ko.wikipedia.org/wiki/GPT_(언어_모델);
snippet: 3 days ago - GPT-4o ("o" for "omni") is a multilingual, multimodal generative pre-trained transformer developed by OpenAI and released in May 2024 as part of its GPT series of models. It can process and generate text, images and audio. Upon release, GPT-4o was free in ChatGPT, though paid subscribers had ..., title: GPT-4o - Wikipedia, link: https://en.wikipedia.org/wiki/GPT-4o;
snippet: April 13, 2026 - OpenAI의 차세대 이미지 모델 GPT-Image-2가 LM Arena에서 세 개의 코드네임으로 유출됐습니다. 텍스트 렌더링 정확도 99%+, 노란 필터 제거, 세계 지식 대폭 강화 등 GPT-Image-1.5 대비 극적 개선을 보여주는 이 모델의 핵심 변화와 실무 ..., title: GPT-Image-2가 유출됐다 — 텍스트 렌더링과 세계 지식이 달라진 OpenAI의 차세대 이미지 모델, link: https://www.working-ref.com/reference/openai-gpt-image-2-leaked-text-rendering-world-knowledge;
snippet: 3 day

In [20]:
from rich.pretty import pprint
pprint(search)

DuckDuckGoSearchResults(
│   api_wrapper=DuckDuckGoSearchAPIWrapper(
│   │   region='wt-wt',
│   │   safesearch='moderate',
│   │   time='y',
│   │   max_results=5,
│   │   backend='auto',
│   │   source='text'
│   ),
│   results_separator=';\n'
)

#### 검색 기능에 옵션 설정하기
- DuckDuckGoSearchAPIWrapper를 활용해 검색 지역과 기간을 설정해보자. 
    - 검색 지역 : "kr-kr"(한국 - 한국어 우선), "wt-wt"(지역, 언어 미 지정), "us-en"(미국 - 영어)
    - 기간 : "w" (최근 일주일 내의 검색 결과만 가져옴), "m"(최근 1개월), "d"(하루)

In [34]:
# DuckDuckGo API wrapper를 사용하여 검색할 때 검색 매개변수를 설정하기 위한 클래스 import
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
wrapper = DuckDuckGoSearchAPIWrapper(region="us-en", time="w")

# 검색 기능을 위한 DuckDuckGoSearchResults 초기화
search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,      # 앞에서 정의한 API wrapper를 사용
    source="text",            # text, news, images 로 검색할 소스 유형 지정
    results_separator=';\n'   # 결과 항목 사이에 구분자 사용 (세미콜론과 줄바꿈)
)

In [35]:
docs = search.invoke("OpenAI의 가장 최신 모델은 무엇인가요?")
print(docs)

snippet: 3 days ago - GPT-4o ("o" for "omni") is a multilingual, multimodal generative pre-trained transformer developed by OpenAI and released in May 2024 as part of its GPT series of models. It can process and generate text, images and audio. Upon release, GPT-4o was free in ChatGPT, though paid subscribers had ..., title: GPT-4o - Wikipedia, link: https://en.wikipedia.org/wiki/GPT-4o;
snippet: 4 days ago - 오픈AI(OpenAI)는 프렌들리 AI를 제고하고 개발함으로써 전적으로 인류에게 이익을 주는 것을 목표로 하는 미국의 인공지능 연구소이다. 이윤을 목적으로 하는 기업 OpenAI LP와 그 모체 조직인 비영리 단체 OpenAI Inc로 구성되어 있다., title: 오픈AI - 위키백과, 우리 모두의 백과사전, link: https://ko.wikipedia.org/wiki/오픈AI;
snippet: 4 days ago - 프로메테우스·카프카·RDBMS에서 데이터를 수집해 아파치 아이스버그 데이터 레이크에 통합하고, 스파크로 변환한 뒤 학습·서빙으로 이어진다. 모델이 바뀌어도 데이터 정제 로직과 학습 파이프라인만 교체하면 나머지는 그대로 쓴다. 모델은 교체 가능한 부품이고 파이프라인이 공통 자산이라는 설명이다., title: [오픈테크넷 2026] 특화 업무에 거대 모델은 과잉…NHN클라우드가 제안한 소형 모델 활용법은?, link: https://www.msn.com/ko-kr/news/other/오픈테크넷-2026-특화-업무에-거대-모델은-과잉-nhn클라우드가-제안한-소형-모델-활용법은/ar-AA2cfm6C;
snippet: 3 d

- 특정 웹 사이트 내에서만 검색 결과를 가져오게 할 수도 있다.

In [36]:
docs = search.invoke("site:https://developers.openai.com/api/docs/models OpenAI의 가장 최신 모델은 무엇인가요?")
print(docs)

snippet: 1 week ago - GPT-6 Astra is our most capable model, built for the hardest end-to-end work. Use it for complex reasoning, coding, computer use, research, and document creation. reasoning.effort supports low, medium, high, xhigh, and max · Get started with ..., title: GPT-6 Astra Model | OpenAI API, link: https://developers.openai.com/api/docs/models/gpt-6-astra;
snippet: 4 days ago - GPT Image 2 is our state-of-the-art image generation model for fast, high-quality image generation and editing. It supports flexible image sizes and high-fidelity image inputs. Learn more in our image generation guide, or see the pricing page ..., title: GPT-Image-2 Model | OpenAI API, link: https://developers.openai.com/api/docs/models/gpt-image-2;
snippet: 20 hours ago - GPT-5.6 Sol is a flagship model in the GPT-5.6 family. It roughly corresponds to the unsuffixed model tier used in earlier GPT-5 families. The gpt-5.6 alias routes requests to GPT-5.6 Sol., title: GPT-5.6 Sol Model | OpenAI API, 

- <참고> 사용할 수 있는 인자값 확인

In [31]:
import inspect
print(inspect.getsource(DuckDuckGoSearchAPIWrapper))

class DuckDuckGoSearchAPIWrapper(BaseModel):
    """Wrapper for DuckDuckGo Search API.

    Free and does not require any setup.
    """

    region: Optional[str] = "wt-wt"
    """
    See https://pypi.org/project/duckduckgo-search/#regions
    """
    safesearch: str = "moderate"
    """
    Options: strict, moderate, off
    """
    time: Optional[str] = "y"
    """
    Options: d, w, m, y
    """
    max_results: int = 5
    backend: str = "auto"
    """
    Options: auto, html, lite
    """
    source: str = "text"
    """
    Options: text, news, images
    """

    model_config = ConfigDict(
        extra="forbid",
    )

    @model_validator(mode="before")
    @classmethod
    def validate_environment(cls, values: Dict) -> Any:
        """Validate that python package exists in environment."""
        try:
            from ddgs import DDGS  # noqa: F401
        except ImportError:
            raise ImportError(
                "Could not import ddgs python package. "
           

- 검색 결과의 링크 확인하기

In [37]:
# 검색 결과의 링크들을 저장할 빈 리스트 초기화
links = []

# 검색 결과를 세미콜론과 줄바꿈 기준으로 분리하고, 각 결과 항목에서 링크를 추출
for doc in docs.split(";\n"):
    print(doc)  # 각 검색 결과 항목을 출력하여 확인
    link = doc.split("link:")[1].strip()  # 각 항목에서 'link:' 이후의 URL 부분만 추출
    links.append(link)  # 추출한 링크를 리스트에 추가

# 모든 링크를 출력
print(links)

snippet: 1 week ago - GPT-6 Astra is our most capable model, built for the hardest end-to-end work. Use it for complex reasoning, coding, computer use, research, and document creation. reasoning.effort supports low, medium, high, xhigh, and max · Get started with ..., title: GPT-6 Astra Model | OpenAI API, link: https://developers.openai.com/api/docs/models/gpt-6-astra
snippet: 4 days ago - GPT Image 2 is our state-of-the-art image generation model for fast, high-quality image generation and editing. It supports flexible image sizes and high-fidelity image inputs. Learn more in our image generation guide, or see the pricing page ..., title: GPT-Image-2 Model | OpenAI API, link: https://developers.openai.com/api/docs/models/gpt-image-2
snippet: 20 hours ago - GPT-5.6 Sol is a flagship model in the GPT-5.6 family. It roughly corresponds to the unsuffixed model tier used in earlier GPT-5 families. The gpt-5.6 alias routes requests to GPT-5.6 Sol., title: GPT-5.6 Sol Model | OpenAI API, li

- 랭체인에서 제공하는 WebBaseLoader 사용하여 링크의 내용 읽어오기
    - uv add bs4

In [39]:
# Langchain의 WebBaseLoader를 사용하여 웹 페이지의 내용을 불러옵니다.
from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader 객체를 생성. 'links'는 웹 페이지의 URL 목록을 담고 있는 변수
# bs_get_text_kwargs는 BeautifulSoup의 get_text() 메소드에 전달될 추가 인자
loader = WebBaseLoader(
    web_paths=links,  # 웹 페이지의 링크 목록을 지정
    bs_get_text_kwargs={
        "strip": True  # 웹 페이지에서 텍스트를 가져올 때 앞뒤의 공백을 제거
    },
)

# 비동기로 웹 페이지의 내용을 로드하고, 각 문서를 page_contents 리스트에 추가
page_contents = []  # 각 웹 페이지의 내용을 저장할 리스트입니다.
async for doc in loader.alazy_load():
    page_contents.append(doc)  # 불러온 문서를 page_contents 리스트에 추가

# page_contents에 있는 각 웹 페이지의 내용을 출력
for content in page_contents:
    print(content)  # 웹 페이지의 내용을 출력
    print('--------------')  # 페이지 구분을 위해 구분선을 출력

page_content='GPT-6 Astra Model | OpenAI APIFor the complete documentation index, seellms.txt. Markdown versions of documentation pages are available by appending.mdto the page URL.ChatGPTHomeAPICodexDocsGuides, concepts, and product docs for CodexUse casesExample workflows and tasks teams can take on with ChatGPT or CodexDocsUse casesTrainingResourcesChatGPTPluginsExtend ChatGPT and CodexWorkspace AgentsTrigger published ChatGPT workspace agentsCommerceBuild commerce flows in ChatGPTAdsPublish and measure ads in ChatGPTResourcesShowcaseDemo apps to get inspiredBlogLearnings and experiences from developersCookbookNotebook examples for building with OpenAI modelsLearnDocs, videos, and demo apps for building with OpenAICommunityPrograms, meetups, and support for buildersStart searchingAPI DashboardTry ChatGPTOverviewModelsAgentsToolsAudio & voiceProductionAPI referenceSearch the API docsSearch docsSuggestedresponses createreasoning_effortrealtimeprompt cachingPrimary navigationAPICodexCh

- 뷰티풀 수프를 이용해 특정 영역만 가져오기
    - 파이썬에서 HTML이나 XML 문서를 쉽게 파싱할 수 있도록 도와주는 라이브러리
    - 개발자 도구를 사용해 본문 영역 찾기. OpenAI Developers 사이트의 경우에는 \<div class="relative"\> 이다.

In [45]:
import requests
from bs4 import BeautifulSoup
import re

# OpenAI Developers 페이지는 상단의 Home/경로 영역과 본문이 분리되어 있음
# 그래서 단일 태그 하나를 고집하지 않고, 여러 후보 중 본문 문단이 제일 많은 영역을 선택

def get_specific_text(url):
    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        selectors = [
            'main article',
            'article',
            'main',
            'div.relative',
            'div[class*=relative]',
            'div.prose',
            '.markdown-body',
            'section',
        ]

        candidates = []
        for sel in selectors:
            candidates.extend(soup.select(sel))

        if not candidates:
            return "본문 내용을 찾을 수 없습니다."

        best_node = None
        best_score = -1

        for node in candidates:
            # 상단 네비게이션, 헤더, 푸터, 광고, 스크립트 제거
            for tag in node.select('nav, header, footer, aside, script, style, iframe, table'):
                tag.decompose()

            paragraphs = node.select('p, li, h1, h2, h3')
            text = node.get_text(' ', strip=True)

            score = len(paragraphs) * 3 + len(text)

            # Home 같은 짧은 네비게이션 텍스트는 제외
            if score > best_score and len(text) > 200:
                best_node = node
                best_score = score

        if best_node is None:
            return "본문 내용을 찾을 수 없습니다."

        text = best_node.get_text('\n', strip=True)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+\n', '\n', text)
        text = re.sub(r'\n[ \t]+', '\n', text)

        return text.strip()

    except requests.exceptions.RequestException as e:
        return f"URL을 가져오는 중 오류 발생: {e}"


# 예시 사용
# article_text = get_specific_text('https://developers.openai.com/api/docs/models/gpt-6-astra')
# print(article_text[:2000])

In [46]:
results = []    # 가져온 내용을 리스트에 담기 위한 변수 선언
for link in links:
    print(f"URL: {link}\n")
    result_text = get_specific_text(link)
    print(f"Content:\n{result_text}")
    print("--------------------------------------------------")
    results.append(result_text)

URL: https://developers.openai.com/api/docs/models/gpt-6-astra

Content:
Models
GPT-6 Astra
Default
Our most capable model, built for the hardest end-to-end work
Our most capable model, built for the hardest end-to-end work
Compare
Try in Playground
Reasoning
Highest
Speed
Fast
Price
$10
•
$50
Input
•
Output
Input
Text, Image
Output
Text
GPT-6 Astra is our most capable model, built for the hardest end-to-end work.
Use it for complex reasoning, coding, computer use, research, and document creation.
reasoning.effort
supports
low
,
medium
,
high
,
xhigh
, and
max
.
Get started with GPT-6 Astra using the
model guide
.
1,050,000 context window
128,000 max output tokens
Apr 30, 2026 knowledge cutoff
Reasoning token support
Pricing
Pricing is based on the number of tokens used, or other metrics based on the model type. For tool-specific models, like search and computer use, there’s a fee per tool call. See details in the
pricing page
.
Text tokens
Per 1M tokens
Input
$10.00
Cached input
$1.00

In [50]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# 대화 기록 객체를 먼저 생성
chat_history = InMemoryChatMessageHistory()

# 참고 문서를 대화 기록으로 설정
if 'results' in globals() and results:
    reference_text = "\n\n".join(str(r) for r in results)
else:
    reference_text = docs

chat_history.add_user_message("다음은 참고 문서입니다.\n" + reference_text)
chat_history.add_user_message("OpenAI의 가장 최신 모델은 무엇인가요?")

# 문서 기반 답변 생성
answer = model.invoke(
    [
        {"role": "system", "content": "다음 문맥을 기반으로 답변해 주세요.\n\n" + reference_text},
        {"role": "user", "content": "OpenAI의 가장 최신 모델은 무엇인가요?"},
    ]
)

# 생성된 답변을 대화 기록에 저장
chat_history.add_ai_message(answer.content)
print(answer.content)

제공된 문맥 기준으로 OpenAI의 가장 최신 모델은 **GPT-6 Astra**입니다.

- **용도:** 복잡한 추론, 코딩, 컴퓨터 사용, 리서치, 문서 작성
- **추론 수준:** low, medium, high, xhigh, max
- **컨텍스트 창:** 1,050,000 토큰
- **최대 출력:** 128,000 토큰
- **지식 기준일:** 2026년 4월 30일
- **API 모델 ID:** `gpt-6-astra`

이미지 생성 분야의 최신 모델은 **GPT-Image-2.5 Flare**입니다.
